# 02. 전처리 & 피처 엔지니어링

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test  = pd.read_csv(DATA_PATH + 'test.csv')

target_col = '임신 성공 여부'  # 실제 타겟 컬럼명으로 수정
print('train:', train.shape, '| test:', test.shape)

## 1. 나이 수치화

In [ ]:
age_map = {
    '만18-34세': 26,
    '만35-37세': 36,
    '만38-39세': 38,
    '만40-42세': 41,
    '만43-44세': 43,
    '만45세 이상': 46
}

for df in [train, test]:
    df['나이_수치'] = df['시술 당시 나이'].map(age_map)

print(train['나이_수치'].value_counts())

## 2. 횟수 컬럼 수치화 (범주형으로 되어있는 경우)

In [ ]:
# 예: '1', '2', '3', '4', '5 이상' 등으로 되어있을 경우 처리
count_cols = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수',
    'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수'
]

def parse_count(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if '이상' in val or '+' in val:
        return 6  # 상한값 처리
    try:
        return float(val)
    except:
        return np.nan

for col in count_cols:
    if col in train.columns:
        train[col + '_num'] = train[col].apply(parse_count)
        test[col + '_num']  = test[col].apply(parse_count)

print('완료')

## 3. 파생 피처 생성 (핵심!)

In [ ]:
for df in [train, test]:
    # 과거 임신 성공률
    total_try = df.get('총 시술 횟수_num', df.get('총 시술 횟수'))
    total_preg = df.get('총 임신 횟수_num', df.get('총 임신 횟수'))
    df['과거_임신성공률'] = total_preg / (total_try + 1e-6)

    # 배아 이식 비율
    if '이식된 배아 수' in df.columns and '총 생성 배아 수' in df.columns:
        df['배아_이식비율'] = df['이식된 배아 수'] / (df['총 생성 배아 수'] + 1e-6)

    # 결측치 여부 피처 (결측 패턴 자체가 정보)
    high_missing_cols = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부', '난자 해동 경과일']
    for col in high_missing_cols:
        if col in df.columns:
            df[col + '_결측여부'] = df[col].isnull().astype(int)

print('파생 피처 생성 완료')
print('새 피처:', [c for c in train.columns if c not in pd.read_csv(DATA_PATH+'train.csv').columns])

## 4. 결측치 처리

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if target_col in num_cols:
    num_cols.remove(target_col)

cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['ID']]

# 수치형: median으로 채우기
for col in num_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col]  = test[col].fillna(median_val)

# 카테고리형: 'Unknown'으로 채우기
for col in cat_cols:
    train[col] = train[col].fillna('Unknown')
    test[col]  = test[col].fillna('Unknown')

print('결측치 처리 완료')
print('train 결측치 합계:', train.isnull().sum().sum())

## 5. 카테고리 인코딩 (Label Encoding)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_dict[col] = le

print('인코딩 완료')

## 6. 전처리 결과 저장

In [ ]:
feature_cols = [c for c in train.columns if c not in ['ID', target_col]]

X_train = train[feature_cols]
y_train = train[target_col]
X_test  = test[feature_cols]

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape)
print('타겟 비율:', y_train.mean().round(4))

# 저장
import pickle
with open('../data/processed.pkl', 'wb') as f:
    pickle.dump({'X_train': X_train, 'y_train': y_train, 'X_test': X_test, 'feature_cols': feature_cols}, f)
print('저장 완료: data/processed.pkl')